## 前提条件

- 4週分の日別PSIモデルを作成し、単品単位での入荷予定作成済み
- 配送は、トラック配送またはケース配送のいずれか
- センターごとに順番に計算するため、生産制約は商品ごとに何日前倒し可能か事前に計算できる（他センターのPSIと委託先の出荷可能数から順次計算）
- ある倉庫×センターに対して、複数の商品を混載する
- *のついた箇所は、モデル化は後から検討

## 変数の定義

##### 集合
- 商品の集合：$P = \{p_1, p_2,\cdots\}$
- 日付の集合：$D = \{d_1, d_2, \cdots\}$

##### 意思決定変数
- ×商品p,日付dの入荷数：$x_{p,d} \quad (p \in P, d \in D)$
- 商品p,日付dの小口(ケース)での入荷数：$x_{p,d}^{(s)} \quad (p \in P, d \in D)$
- 商品p,日付dの大口(トラック)での入荷数：$x_{p,d}^{(l)} \quad (p \in P, d \in D)$
- 日付dのトラック台数：$n_d$


##### 入力データ
- 商品p,日付dのベース入荷数：$\bar{x}_{p,d}$
- 商品p,日付dの出荷予測：$s_{p,d}$
- 前日末在庫:$I_{p,0}$
- *商品pの今回の発注の納品日：$t_p$
- *ある日付までに出荷可能な数量（累積）：$c_{p,d}$
    - 出荷可能数の累積から、他センターのPSIを減算した結果を入荷日ベースにしたもの



#### 従属変数
- 商品p,日付dの在庫：$I_{p,d}$

#### 定数
- 商品pを1ケース1日保管する在庫コスト：$h_p$
- 商品pをケースで運んだ際のケース当たり単価：$c_p^s$
- トラック1台当たりの単価：$C^l$
- 商品pのみでトラック配送した際、何ケースで満車となるか：$K_p$

### 制約条件
- 在庫推移（初日）：$I_{p,1} = I_{p,0}-s_{p,1}+x_{p,1}$
- 在庫推移（2日目以降）：$I_{p,t} = I_{p,t-1}-s_{p,t}+x_{p,t} \quad (t \ne t_1)$
- ベースの入荷数以上の発注数：$\sum_{d_1}^{d_i} (x_{p,d}^{(s)} + x_{p,d}^{(l)}) \ge \sum_{\tau = d_1}^{d_i} \bar{x}_{p,d} \quad (p \in P, d_i \in D)$
    - **合計を一致させる制約は必要？
- トラック配送キャパ：$\sum_p \frac{x_{p,d}^{(l)}}{K_p} \le n_d \quad (d \in D)$
- *今回発注納品日以前の入荷数は変更不可：$x_{p,d} = 0 \quad (d \le t_p)$
- *入荷数の累積≤出荷可能数の累積：$\sum_{d_1}^{d_i}x_{p,d} \le \sum_{d_1}^{d_i}c_{p,d} \quad (p \in P, d_i \in D)$


### 目的関数
- 在庫コスト：$\sum_{p,d} h_p I_{p,d}$
- 配送コスト(小口分)：$\sum_{p,d} c_px_{p,d}^{(s)}$
- 配送コスト(大口分)：$\sum_d C^l n_d$




In [2]:
import pandas as pd
import pulp

In [3]:
dfs = pd.read_excel('input/InputData.xlsx', sheet_name = None)
product_master_df = dfs['商品マスタ']
parameters_df = dfs['パラメータ']
time_series_data_df = dfs['時系列データ']
inventory_init_df = dfs['前日末在庫']

In [4]:
# 集合
products_list = (time_series_data_df['商品コード']
                 .drop_duplicates()
                 .sort_values()
                 .to_list())
days_list = (time_series_data_df['日付']
                 .drop_duplicates()
                 .sort_values()
                 .to_list())

In [5]:
# 定数
holding_cost_dict = (product_master_df
    .set_index('商品コード')
    ['在庫コスト']
    .to_dict())
cost_per_case_dict = (product_master_df
    .set_index('商品コード')
    ['配送費_ケース']
    .to_dict())
cost_per_truck = (parameters_df
                  .set_index('項目')
                  .loc['配送費_車両','数量'])
max_cases_per_truck_single_product_dict = (product_master_df
    .set_index('商品コード')
    ['CS/車両']
    .to_dict())

In [6]:
# 入力データ
x_bar_dict = (time_series_data_df
              .set_index(['商品コード', '日付'])
              ['入荷予定']
              .fillna(0)
              .to_dict())
shipping_forecast_dict = (time_series_data_df
                          .set_index(['商品コード', '日付'])
                          ['出荷予測']
                          .fillna(0)
                          .to_dict())
inventory_init_dict = (inventory_init_df
                            .set_index('商品コード')
                            ['前日末在庫']
                            .to_dict())

In [7]:
# -----------------------------
# モデル
# -----------------------------
model = pulp.LpProblem("Delivery_Stock_Optimization", pulp.LpMinimize)

In [8]:
# -----------------------------
# 変数
# -----------------------------
x_small = (pulp.LpVariable.dicts
           ('x_small',
            (products_list, days_list),
            lowBound=0))
x_large = (pulp.LpVariable.dicts
           ('x_large',
            (products_list, days_list),
            lowBound=0))
n_trucks = (pulp.LpVariable.dicts
            ('n_trucks',
             days_list,
             lowBound=0,
             cat='Integer'))
I = (pulp.LpVariable.dicts
     ("Inventory",
      (products_list, days_list),
      lowBound=0))

In [9]:
# -----------------------------
# 目的関数
# -----------------------------

### 在庫コストの追加必要
model += (
     pulp.lpSum(cost_per_case_dict[p] * x_small[p][d] for p in products_list for d in days_list)
    + pulp.lpSum(cost_per_truck * n_trucks[d] for d in days_list))

In [10]:
# -----------------------------
# 制約条件
# -----------------------------
for d_idx, d in enumerate(days_list):
    # トラック容量（混載）
    model += (pulp.lpSum(x_large[p][d] 
                         / max_cases_per_truck_single_product_dict[p]
                         for p in products_list) 
              <= n_trucks[d])
    # ベースの入荷数以上の発注数
    model += (pulp.lpSum(x_small[p][d] + x_large[p][d] for p in products_list)
              >= pulp.lpSum(x_bar_dict[(p, d)] for p in products_list))

    for p in products_list:
        # 在庫推移
        if d_idx == 0:
            model += (I[p][d]
                      ==
                      inventory_init_dict[p]
                      + x_small[p][d]
                      + x_large[p][d]
                      - shipping_forecast_dict[(p, d)])
        else:
            d_prev = days_list[d_idx - 1]
            model += (I[p][d]
                      ==
                      I[p][d_prev]
                      + x_small[p][d]
                      + x_large[p][d]
                      - shipping_forecast_dict[(p, d)])
        # ベースの入荷予定以上の発注数（前倒しのみ可能）
        tau_list = days_list[:d_idx+1]
        model += (pulp.lpSum(x_small[p][tau]
                             + x_large[p][tau]
                             for tau in tau_list)
                  >=
                  pulp.lpSum(x_bar_dict[(p, tau)] 
                             for tau in tau_list))

In [21]:
# -----------------------------
# 求解
# -----------------------------
model.solve()

# -----------------------------
# 出力
# -----------------------------
print("Status:", pulp.LpStatus[model.status])
print("Total Cost:", pulp.value(model.objective))

Status: Optimal
Total Cost: 182480.0


In [16]:
# 結果の整形
decision_variables = []
for d in days_list:
    for p in products_list:
        decision_variables.append({
            "d": d,
            "p": p,
            "x_small": x_small[p][d].value(),
            "x_large": x_large[p][d].value(),
            "inventory": I[p][d].value()
        })
trucks = []
for d in days_list:
    trucks.append({
        "d": d,
        "n_trucks": n_trucks[d].value()
    })

In [ ]:
# DataFrame化
decision_variables_df = pd.DataFrame(decision_variables)
trucks_df = pd.DataFrame(trucks)

# エクセルに出力
decision_variables_df.to_excel('output/decision_variables.xlsx', index=False)
trucks_df.to_excel('output/trucks.xlsx', index=False)
